# Week 6 Lab: Instrumental Variables and Return to Schooling

## Learning Objectives

By the end of this lab, you will be able to:
1. Implement heteroskedasticity-robust standard errors
2. Understand the endogeneity problem in wage regressions
3. Estimate reduced form regressions to assess instrument relevance
4. Implement IV estimation using three equivalent approaches

## Background: Card's (1993) Return to Schooling

### The Structural Equation

In Week 5, we estimated the Mincer wage equation:

$$\ln(\text{earn}) = \beta_1 + \beta_2 \cdot \text{educ} + \beta_3 \cdot \text{exper} + \beta_4 \cdot \text{expersq} + \cdots + u$$

where $\beta_2$ is the **return to schooling**—the percentage increase in earnings from one additional year of education.

### The Endogeneity Problem

Labor economists believe `educ` is **endogenous** because:

1. **Ability bias**: More able individuals tend to get more education *and* earn higher wages regardless of education
   - If ability ($A$) is in the error term: $\text{Cov}(\text{educ}, u) \neq 0$ because $\text{Cov}(\text{educ}, A) \neq 0$

2. **Measurement error**: Self-reported education may be measured with error

3. **Reverse causality**: Expected future earnings may influence education decisions

**Consequence**: OLS is **biased and inconsistent** for $\beta_2$. The direction of bias is typically upward (ability bias makes OLS overestimate the return to schooling).

## Setup and Data Loading

In [ ]:
using Downloads, DelimitedFiles, LinearAlgebra, Printf, Statistics

# Load Card's (1993) data
url = "https://raw.githubusercontent.com/juergenmeinecke/EMET8014/main/datasets/card.csv"
data = readdlm(Downloads.download(url), ',');

# Extract dependent variable (log wages)
Y = Vector{Float64}(data[:, 33])
N = length(Y)
println("Sample size: N = $N")

In [ ]:
# Construct regressor matrix X
# Column order: constant, educ, exper, expersq, black, south, smsa, smsa66, reg661-reg668

const_term = ones(N)
educ = data[:, 4]           # Years of education (ENDOGENOUS)
exper = data[:, 32]         # Experience
expersq = data[:, 34] ./ 100  # Experience squared / 100
black = data[:, 22]         # Race dummy
south = data[:, 24]         # South dummy  
smsa = data[:, 23]          # Metropolitan area dummy
smsa66 = data[:, 25]        # Metropolitan area in 1966
region_dummies = data[:, 12:19]  # Regional dummies

X = Matrix{Float64}(hcat(const_term, educ, exper, expersq, black, south, smsa, smsa66, region_dummies))
N, K = size(X)

println("Number of regressors: K = $K")
println("Regressor names: constant, educ, exper, expersq, black, south, smsa, smsa66, reg661-reg668")

## Exercise 1: OLS with Heteroskedasticity-Robust Standard Errors

### The Asymptotic Distribution of OLS

Under standard assumptions (but allowing for heteroskedasticity):

$$\sqrt{N}(\hat{\beta}^{\text{OLS}} - \beta) \xrightarrow{d} N(0, \Omega)$$

where the **sandwich formula** for $\Omega$ is:

$$\Omega = E(X_i X_i')^{-1} \cdot E(u_i^2 X_i X_i') \cdot E(X_i X_i')^{-1}$$

### Estimation

We estimate $\Omega$ using the sample analog with a degrees-of-freedom correction:

$$\begin{aligned}
    \hat{\Omega}_{OLS} 
        :=& \left(\tfrac{1}{N} \sum_{i=1}^N X_i X_i'\right)^{-1} 
            \left(\frac{1}{N-K} \sum_{i=1}^N \hat{u}_i^2 X_i X_i'\right) 
            \left(\tfrac{1}{N} \sum_{i=1}^N X_i X_i'\right)^{-1} \\
        =& \left(\tfrac{1}{N} X'X\right)^{-1} 
            \left(\tfrac{1}{N-K} X' D X\right) 
            \left(\tfrac{1}{N} X'X\right)^{-1}\\
      \text{where } D &:= \text{diag}(\hat{u}_1^2, \ldots, \hat{u}_N^2)
\end{aligned}$$

<div class="alert alert-success">

**Key Result:** 

The variance of $\hat{\beta}^{\text{OLS}}$ is then approximately $\hat{\Omega}_{OLS}/N = \frac{N}{N-K} (X'X)^{-1} X' D X (X'X)^{-1}$.
    
</div>



**Note:** This is sometimes called the "Huber-White" or "Eicker-Huber-White" estimator.

#### Your task

Write a function `lm_ols` (*lm* for linear model) that takes the arguments `Y` and `X` and returns

* a vector containing the OLS estimator;

* a matrix corresponding to $\widehat{\Omega}/N$ from the lecture (the estimated asymptotic covariance matrix of $\widehat{\beta}^{OLS}$)

(Throughout this entire notebook always allow for heteroskedasticity.)


In [ ]:
"""
    lm_ols(Y, X)

Compute OLS estimates with heteroskedasticity-robust standard errors.

# Arguments
- `Y`: N×1 vector of dependent variable
- `X`: N×K matrix of regressors (should include constant)

# Returns
- `β_hat`: K×1 vector of coefficient estimates
- `Ω_hat/N`: K×K estimated asymptotic variance of β̂ (robust)

# Notes
Uses the sandwich formula with degrees-of-freedom correction (HC1):
    Ω̂ /N = (N/(N-K)) (X'X)⁻¹ X'DX (X'X)⁻¹
where D = diag(û₁², ..., ûₙ²).
"""
function lm_ols(Y::Vector{Float64}, X::Matrix{Float64})
    N, K = size(X)
    
    β_hat = nothing # YOUR CODE HERE
    u_hat = nothing # YOUR CODE HERE
    
    # Sandwich formula: bread * meat * bread
    bread = nothing # YOUR CODE HERE
    meat = nothing # YOUR CODE HERE
    Ω_hat_over_N = nothing # YOUR CODE HERE
    
    return β_hat, Ω_hat_over_N
end


Now use your function to obtain the actual estimates.

In [ ]:
# Estimate and report
β_ols, Omega_ols = nothing # YOUR CODE HERE

# Focus on the return to schooling (β₂)
se_β2 = nothing # YOUR CODE HERE
t_stat = nothing # YOUR CODE HERE

# Uncomment when ready:
# @printf "OLS Results for Return to Schooling (β₂):\n"
# @printf "  Point estimate: %.4f\n" β_ols[2]
# @printf "  Robust SE:      %.5f\n" se_β2
# @printf "  t-statistic:    %.2f\n" t_stat
# @printf "\nInterpretation: One additional year of education is associated with\n"
# @printf "a %.1f%% increase in wages (under OLS, which may be biased).\n" (100 * β_ols[2])

## Card's Instrumental Variable: Proximity to College

### The Idea

To address endogeneity, Card proposes using **proximity to a 4-year college** as an instrument:

$$\text{nearc4} = \begin{cases} 1 & \text{if person lives near a 4-year college} \\ 0 & \text{otherwise} \end{cases}$$

### Instrument Validity Requires Two Conditions

1. **Relevance**: $\text{Cov}(\text{nearc4}, \text{educ}) \neq 0$
   - Living near a college should affect education attainment (lower cost of attendance)
   
2. **Exogeneity** (Exclusion restriction): $\text{Cov}(\text{nearc4}, u) = 0$
   - Proximity should affect wages *only* through education, not directly
   - This is more controversial (see Card's paper for discussion)

### Potential Violations of Exogeneity

- Families who value education may choose to live near colleges
- College towns may have different labor markets
- College proximity may be correlated with urban/rural differences

Card includes many control variables to address these concerns.

## Exercise 2: Define X1, X2, Z1, Z2 Matrices

Following the lecture notation, we partition the regressors and instruments:

- $X = (X_1, X_2)$ where $X_2$ is endogenous (education)
- $Z = (Z_1, Z_2)$ where $Z_1 = X_1$ (exogenous regressors serve as their own instruments) and $Z_2$ is the new instrument

In [ ]:
# Partition the regressors
# X1: exogenous regressors (constant, exper, expersq, black, south, smsa, smsa66, region dummies)
# X2: endogenous regressor (education)

# Hint: you need to select all columns of X except the endogenous one (column 2)
X1 = nothing  # YOUR CODE HERE
X2 = nothing  # YOUR CODE HERE

# Construct instrument matrix
# Z1: exogenous regressors are their own instruments
# Z2: new instrument (nearc4 = proximity to 4-year college, column 3 in data)

Z1 = nothing  # YOUR CODE HERE
Z2 = nothing  # YOUR CODE HERE
Z = nothing   # YOUR CODE HERE

# Uncomment when ready:
# println("Dimensions:")
# println("  X1: $(size(X1)) - Exogenous regressors")
# println("  X2: $(length(X2)) - Endogenous regressor (education)")
# println("  Z:  $(size(Z))  - Full instrument matrix")

## Exercise 3: First-Stage Regression

### Testing Instrument Relevance

The **first-stage regression** regresses the endogenous variable on all instruments:

$$\text{educ}_i = Z_i' \pi + v_i$$

We're particularly interested in $\pi_{16}$, the coefficient on `nearc4`. A significant coefficient indicates the instrument is **relevant**.

**Rule of thumb**: The first-stage F-statistic for the excluded instruments should exceed 10 to avoid weak instrument problems.

In [ ]:
# First-stage regression: education on all instruments
# Use your lm_ols function
π_hat, Omega_π = nothing  # YOUR CODE HERE

# The coefficient on nearc4 (last element)
π_hat_nearc4 = nothing  # YOUR CODE HERE
se_nearc4 = nothing     # YOUR CODE HERE
t_nearc4 = nothing      # YOUR CODE HERE

# Uncomment when ready:
# @printf "First-Stage Regression Results:\n"
# @printf "\nEffect of nearc4 on education (π_16):\n"
# @printf "  Point estimate: %.3f\n" π_hat_nearc4
# @printf "  Robust SE:      %.3f\n" se_nearc4
# @printf "  t-statistic:    %.2f\n" t_nearc4
# @printf "\nInterpretation: Living near a 4-year college is associated with\n"
# @printf "approximately %.2f additional years of education.\n" π_hat_nearc4

## Exercise 4: Reduced Form for Y

The **reduced form** regresses the outcome directly on instruments:

$$Y_i = Z_i' \lambda + w_i$$

The coefficient $\lambda_2$ on `nearc4` captures the **total effect** of college proximity on wages (working through education).

**Key insight**: The IV estimator can be expressed as $\hat{\beta}_2^{IV} = \hat{\lambda}_2 / \hat{\pi}_2$ (the Wald estimator in the just-identified case).

In [ ]:
# Reduced form regression: Y on Z
λ_hat, Omega_λ = nothing  # YOUR CODE HERE

λ_hat_nearc4 = nothing  # YOUR CODE HERE
se_λ = nothing          # YOUR CODE HERE
t_λ = nothing           # YOUR CODE HERE

# Uncomment when ready:
# @printf "Reduced Form Results:\n"
# @printf "\nEffect of nearc4 on log wages (λ_16):\n"
# @printf "  Point estimate: %.3f\n" λ_hat_nearc4
# @printf "  Robust SE:      %.3f\n" se_λ
# @printf "  t-statistic:    %.2f\n" t_λ

In [ ]:
# Verify Wald estimator relationship
# Hint: the Wald estimator is a ratio of two reduced-form coefficients
β2_wald = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "\nWald estimator (ratio of reduced forms): %.4f\n" β2_wald
# @printf "This should equal the IV estimate we compute below.\n"

## Exercise 5: Three Equivalent IV Estimators

We'll verify that three different approaches give the same answer:

### Approach 1: Direct Formula
$$\hat{\beta}^{IV} = (Z'X)^{-1} Z'Y$$

### Approach 2: Two-Stage Least Squares (2SLS) - Traditional
1. Regress $X$ on $Z$ to get $\hat{X} = Z(Z'Z)^{-1}Z'X$ (fitted values)
2. Regress $Y$ on $\hat{X}$

### Approach 3: Control Function / Hausman Test Setup
1. Regress $X_2$ on $Z$ to get residuals $\hat{v}$
2. Regress $Y$ on $X$ and $\hat{v}$
3. Coefficient on $X_2$ is the IV estimate

**Why three approaches?** Each gives different intuition and the third provides a test for endogeneity (Hausman test).

In [ ]:
# Approach 1: Direct IV formula
# Note: This only works when dim(Z) = dim(X) (just-identified case)
# Hint: translate the formula from the text above into Julia
β_hat_iv1 = nothing  # YOUR CODE HERE
β2_iv1 = nothing     # YOUR CODE HERE (extract the education coefficient)

# Uncomment when ready:
# @printf "Approach 1 (Direct formula): β2 = %.6f\n" β2_iv1

In [ ]:
# Approach 2: Two-Stage Least Squares
# Stage 1: Project X onto the column space of Z to get fitted values
# Hint: use the backslash operator for efficiency
Pi_hat = nothing  # YOUR CODE HERE
X_hat = nothing   # YOUR CODE HERE (fitted values from Stage 1)

# Stage 2: Regress Y on X_hat
β_hat_iv2 = nothing  # YOUR CODE HERE
β2_iv2 = nothing     # YOUR CODE HERE

# Uncomment when ready:
# @printf "Approach 2 (2SLS):           β2 = %.6f\n" β2_iv2

In [ ]:
# Approach 3: Control Function / Residual Inclusion
# Stage 1: Regress X2 on Z and obtain the first-stage residuals
π_hat_cf = nothing  # YOUR CODE HERE
v_hat = nothing     # YOUR CODE HERE (first-stage residuals)

# Stage 2: Augment the original X with the first-stage residuals, then run OLS
X_augmented = nothing  # YOUR CODE HERE
β_hat_cf = nothing     # YOUR CODE HERE
β2_iv3 = nothing       # YOUR CODE HERE (extract the education coefficient)

# Uncomment when ready:
# @printf "Approach 3 (Control function): β2 = %.6f\n" β2_iv3

In [ ]:
# Verify all three are (numerically) identical
# Uncomment when ready:
# @printf "\nDifferences (should be ~1e-12 or smaller):\n"
# @printf "  |Approach 1 - Approach 2| = %.2e\n" abs(β2_iv1 - β2_iv2)
# @printf "  |Approach 1 - Approach 3| = %.2e\n" abs(β2_iv1 - β2_iv3)
# @printf "  |Approach 2 - Approach 3| = %.2e\n" abs(β2_iv2 - β2_iv3)

## Summary

### What We Learned

1. **Heteroskedasticity-robust SEs**: Use the sandwich formula to get valid inference even with non-constant error variance

2. **Endogeneity**: Education in wage regressions is likely endogenous due to ability bias

3. **Instrumental Variables**: `nearc4` (proximity to college) serves as an instrument because:
   - It predicts education (relevance): coefficient approximately 0.32 additional years
   - It's arguably exogenous (conditional on controls)

4. **Three equivalent IV approaches**: Direct formula, 2SLS, and control function all give the same point estimate

### Looking Ahead

In Week 7, we'll use Monte Carlo simulations to:
- Study the finite-sample properties of OLS and IV
- Understand how endogeneity affects OLS bias
- Explore the bias-variance trade-off in IV estimation